# Image Spliter

## Imports

In [134]:
#installs
%pip install pillow
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [135]:
#imports
from PIL import Image
import numpy as np
import os
import re
from pathlib import Path
import shutil
import random

## Normalizing names
File names comes with no standard naming convention and thus it is difficult to access them with code.\
The first step must be to give them standardized names so splitting is also easier.

In [136]:
DAMAGED_PATH = "paired_dataset_art/damaged"
UNDAMAGED_PATH = "paired_dataset_art/undamaged"

In [137]:
def get_files(directory_path):
    """
    Gets and returns a file path list from the specified directory.
    """
    directory = Path(directory_path)
    if not directory.exists():
        raise FileNotFoundError(f"Directory does not exist: {directory_path}")
    if not directory.is_dir():
        raise NotADirectoryError(f"Not a directory: {directory_path}")

    files = [p for p in directory.iterdir() if p.is_file()]
    files.sort(key=lambda p: p.name.casefold())  # Finder-like alphabetical (case-insensitive)

    filespath_list = [str(p.resolve()) for p in files]
    return filespath_list

In [138]:
damaged_list = get_files(DAMAGED_PATH)
display(len(damaged_list)) #sanity check

undamaged_list = get_files(UNDAMAGED_PATH)
display(len(undamaged_list)) #sanity check

113

113

In [139]:
display(damaged_list)

['/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/07_RESTORATION_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/alex_before.jpg',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/and_Salvador_Mundi_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/and_treatment_oil_on_canvas_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/angel.jpg',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/ArchimedesConservation_05_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/arena_before.jpg',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paire

In [140]:
display(undamaged_list)

['/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/undamaged/07_RESTORATION_after.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/undamaged/_after.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/undamaged/alex_after.jpg',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/undamaged/and_Salvador_Mundi_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/undamaged/and_treatment_oil_on_canvas_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/undamaged/angel_after.jpg',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/undamaged/ArchimedesConservation_05_after.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/undamaged/arena_after.jpg',
 '/Users/matt/Documents/Local Programming/DeepLearni

In [141]:
def remove_doubles(list):
    """
    This function scours the specified list to removes possible doubles (based on names).\n
    It doesn't take the file extension into account (so if there is a 'file.jpg' and 'file.png', one of them will get removed)
    """
    seen = set()
    filtered = []

    for item in list:
        filename = os.path.basename(str(item))
        name, _ext = os.path.splitext(filename)

        if name not in seen:
            seen.add(name)
            filtered.append(item)

    return filtered

In [142]:
from PIL import Image, ImageOps
import numpy as np

def content_bbox(img: Image.Image, threshold=100):
    """
    Returns (left, upper, right, lower) bbox of non-white pixels.
    Works for RGB/RGBA.
    """
    arr = np.array(img)
    if arr.ndim == 3 and arr.shape[2] == 4:
        rgb = arr[..., :3]
        a = arr[..., 3]
        mask = (a > 0) & (rgb < threshold).any(axis=2)
    else:
        if arr.ndim == 2:
            arr = arr[..., None]
        mask = (arr < threshold).any(axis=2)

    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return (0, 0, img.size[0], img.size[1])
    return (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)


def crop_pair_same_box(damaged_path, undamaged_path, threshold=100):
    """
    Crops BOTH images using the bbox computed from the UNDAMAGED image.
    This prevents damage artifacts from changing the crop box and shifting tiles.
    Overwrites files in-place.
    """
    with Image.open(undamaged_path) as u_img:
        u_img = u_img.convert("RGBA")  # safe for bbox
        box = content_bbox(u_img, threshold=threshold)

    # Apply same crop to both
    for p in [damaged_path, undamaged_path]:
        with Image.open(p) as img:
            img = img.convert("RGB")
            cropped = img.crop(box)
            cropped.save(p)

In [143]:
def crop_white_borders(image_paths, threshold=100):
    """
    Crops exterior near-white borders for every image path in image_paths (in-place overwrite).
    If the content edge is circular/spherical, this still crops to the tightest rectangle
    enclosing all non-white pixels (so you don't end up with empty/null crops).
    """
    for image_path in image_paths:
        image_path = Path(image_path)

        if not image_path.is_file():
            continue

        with Image.open(image_path) as img:
            arr = np.array(img)

            # Build a "non-white" mask.
            # - If RGBA: also treat transparent pixels as "white background"
            # - Else: use RGB/gray intensity threshold
            if arr.ndim == 3 and arr.shape[2] == 4:
                rgb = arr[..., :3].astype(np.float32)
                alpha = arr[..., 3].astype(np.float32)
                gray = rgb.mean(axis=2)
                mask = (gray < threshold) & (alpha > 0)
            elif arr.ndim == 3:
                gray = arr[..., :3].astype(np.float32).mean(axis=2)
                mask = gray < threshold
            else:
                gray = arr.astype(np.float32)
                mask = gray < threshold

            # If nothing non-white, keep original (avoid empty crop)
            if not mask.any():
                continue

            coords = np.argwhere(mask)
            y0, x0 = coords.min(axis=0)
            y1, x1 = coords.max(axis=0) + 1

            cropped = img.crop((x0, y0, x1, y1))

            # Overwrite while keeping format consistent with extension
            ext = image_path.suffix.lower()
            if ext in (".jpg", ".jpeg"):
                cropped = cropped.convert("RGB")  # JPEG can't save alpha
                cropped.save(image_path, quality=95, optimize=True)
            else:
                cropped.save(image_path)

            cropped.close()

In [144]:
for dp, up in zip(damaged_list, undamaged_list):
    crop_pair_same_box(dp, up, threshold=100)

In [145]:
def pad_to_multiple(img: Image.Image, tile_size: int, fill=(255, 255, 255)):
    w, h = img.size
    new_w = ((w + tile_size - 1) // tile_size) * tile_size
    new_h = ((h + tile_size - 1) // tile_size) * tile_size
    pad_right = new_w - w
    pad_bottom = new_h - h
    if pad_right == 0 and pad_bottom == 0:
        return img
    return ImageOps.expand(img, border=(0, 0, pad_right, pad_bottom), fill=fill)

In [146]:
filtered_damaged_list = remove_doubles(damaged_list)
display(len(filtered_damaged_list)) #sanity check

filtered_undamaged_list = remove_doubles(undamaged_list)
display(len(filtered_undamaged_list)) #sanity check

108

108

In [147]:
display(filtered_damaged_list)


['/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/07_RESTORATION_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/alex_before.jpg',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/and_Salvador_Mundi_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/and_treatment_oil_on_canvas_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/angel.jpg',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/ArchimedesConservation_05_before.png',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paired_dataset_art/damaged/arena_before.jpg',
 '/Users/matt/Documents/Local Programming/DeepLearning_Artworks/paire

Image resizing step:

Some of the images have a pixel dimension difference between damaged and undamaged.\
Bellow, we make the biggest image take the dimensions of the smallest.

This part is important so the tiles matchup correctly further down the line.

In [148]:
def resize_pairs_to_smallest(damaged_list, undamaged_list, resample=Image.Resampling.LANCZOS):
    """
    Iterates through damaged_list and undamaged_list in parallel and resizes whichever
    image is larger so both end up with the smallest (w,h) of the pair.
    Overwrites files in place.
    """
    if len(damaged_list) != len(undamaged_list):
        raise ValueError("damaged_list and undamaged_list must have the same length")

    for i in range(len(damaged_list)):
        d_path = damaged_list[i]
        u_path = undamaged_list[i]

        with Image.open(d_path) as d_img, Image.open(u_path) as u_img:
            dw, dh = d_img.size
            uw, uh = u_img.size

            target = (min(dw, uw), min(dh, uh))

            if (dw, dh) != target:
                d_img.resize(target, resample=resample).save(d_path)

            if (uw, uh) != target:
                u_img.resize(target, resample=resample).save(u_path)

In [149]:
resize_pairs_to_smallest(filtered_damaged_list,filtered_undamaged_list)

In [150]:
import os
import shutil

def assign_index_bichannel(damaged_list,undamaged_list,output_directory):
    """
    Assigns a common index to the couple damaged/undamaged images.\n
    This function assumes the corresponding damaged and undamaged images come at the same list index.

    Example of index formating : 001....024...174....999
    """
    if len(damaged_list) != len(undamaged_list):
        raise ValueError("damaged_list and undamaged_list must have the same length")

    damaged_out = os.path.join(output_directory, "damaged")
    undamaged_out = os.path.join(output_directory, "undamaged")
    os.makedirs(damaged_out, exist_ok=True)
    os.makedirs(undamaged_out, exist_ok=True)

    n = len(damaged_list)
    width = max(3, len(str(n)))  # 001..999 minimum, grows if needed

    indexed_pairs = []
    for i, (dmg_path, undmg_path) in enumerate(zip(damaged_list, undamaged_list), start=1):
        idx = str(i).zfill(width)

        dmg_ext = os.path.splitext(str(dmg_path))[1]
        undmg_ext = os.path.splitext(str(undmg_path))[1]

        dmg_dst = os.path.join(damaged_out, f"{idx}{dmg_ext}")
        undmg_dst = os.path.join(undamaged_out, f"{idx}{undmg_ext}")

        shutil.copy2(dmg_path, dmg_dst)
        shutil.copy2(undmg_path, undmg_dst)

        indexed_pairs.append((idx, dmg_dst, undmg_dst))

    return indexed_pairs

In [151]:
INDEXED_DIRECTORY = "indexed_paired_images"

In [152]:
indexed_pairs = assign_index_bichannel(filtered_damaged_list,filtered_undamaged_list,INDEXED_DIRECTORY)
display(indexed_pairs)

[('001',
  'indexed_paired_images/damaged/001.png',
  'indexed_paired_images/undamaged/001.png'),
 ('002',
  'indexed_paired_images/damaged/002.png',
  'indexed_paired_images/undamaged/002.png'),
 ('003',
  'indexed_paired_images/damaged/003.jpg',
  'indexed_paired_images/undamaged/003.jpg'),
 ('004',
  'indexed_paired_images/damaged/004.png',
  'indexed_paired_images/undamaged/004.png'),
 ('005',
  'indexed_paired_images/damaged/005.png',
  'indexed_paired_images/undamaged/005.png'),
 ('006',
  'indexed_paired_images/damaged/006.jpg',
  'indexed_paired_images/undamaged/006.jpg'),
 ('007',
  'indexed_paired_images/damaged/007.png',
  'indexed_paired_images/undamaged/007.png'),
 ('008',
  'indexed_paired_images/damaged/008.jpg',
  'indexed_paired_images/undamaged/008.jpg'),
 ('009',
  'indexed_paired_images/damaged/009.png',
  'indexed_paired_images/undamaged/009.jpg'),
 ('010',
  'indexed_paired_images/damaged/010.png',
  'indexed_paired_images/undamaged/010.png'),
 ('011',
  'indexed_

## Dataset splitting
Before splitting individual images, we should make the different sets.\
Split is:\
80/20 Train+Test / Evaluation sets\
of those 80%, we do 70/30 Train/Test

In [153]:
#Config for set splitting.
TRAIN_EVAL_SPLIT = (80,20)
SUB_TRAIN_TEST_SPLIT = (70,30)

TRAIN_SET_PATH = "splitsets/train_indexed_images"
TEST_SET_PATH = "splitsets/test_indexed_images"
EVAL_SET_PATH = "splitsets/eval_indexed_images"

In [154]:
def split_train_eval(index_paired, train_eval_split=TRAIN_EVAL_SPLIT, seed=42):
    """
    Phase 1:
    80/20 split between (train+test) and eval.

    index_paired items must look like: (idx, damaged_path, undamaged_path)
    """
    pairs = list(index_paired)

    # Shuffle pairs together so damaged/undamaged stay aligned by idx
    rng = random.Random(seed)
    rng.shuffle(pairs)

    n = len(pairs)
    total = train_eval_split[0] + train_eval_split[1]
    eval_n = int(n * (train_eval_split[1] / total))  # 20%
    train_test_n = n - eval_n                        # 80%

    train_test_pairs = pairs[:train_test_n]
    eval_pairs = pairs[train_test_n:]

    return train_test_pairs, eval_pairs

In [155]:
def split_train_test(train_test_pairs, sub_train_test_split=SUB_TRAIN_TEST_SPLIT, seed=42):
    """
    Phase 2:
    70/30 split between train and test (within the train+test set).

    items are: (idx, damaged_path, undamaged_path)
    """
    pairs = list(train_test_pairs)

    rng = random.Random(seed)
    rng.shuffle(pairs)

    n = len(pairs)
    total = sub_train_test_split[0] + sub_train_test_split[1]
    test_n = int(n * (sub_train_test_split[1] / total))  # 30%
    train_n = n - test_n                                 # 70%

    train_pairs = pairs[:train_n]
    test_pairs = pairs[train_n:]

    return train_pairs, test_pairs

In [156]:
def move_split_files(train_pairs, test_pairs, eval_pairs,
                     train_out=TRAIN_SET_PATH, test_out=TEST_SET_PATH, eval_out=EVAL_SET_PATH):
    """
    Phase 3:
    Move files into split directories, ensuring damaged and undamaged keep the SAME idx.

    Destination filenames are forced to be:
      <idx><original_extension>
    so both images share the same index in their respective subfolders.

    index_paired format:
      (idx, damaged_path, undamaged_path)
    """
    def _prepare(root):
        root = Path(root)
        (root / "damaged").mkdir(parents=True, exist_ok=True)
        (root / "undamaged").mkdir(parents=True, exist_ok=True)
        return root

    train_root = _prepare(train_out)
    test_root  = _prepare(test_out)
    eval_root  = _prepare(eval_out)

    def _move_subset(pairs, root):
        for idx, dmg_path, undmg_path in pairs:
            dmg_path = Path(dmg_path)
            undmg_path = Path(undmg_path)

            dmg_ext = dmg_path.suffix
            und_ext = undmg_path.suffix

            dmg_dst = root / "damaged" / f"{idx}{dmg_ext}"
            und_dst = root / "undamaged" / f"{idx}{und_ext}"

            shutil.move(str(dmg_path), str(dmg_dst))
            shutil.move(str(undmg_path), str(und_dst))

    _move_subset(train_pairs, train_root)
    _move_subset(test_pairs,  test_root)
    _move_subset(eval_pairs,  eval_root)

    return {
        "train": str(train_root),
        "test": str(test_root),
        "eval": str(eval_root),
        "counts": {
            "train": len(train_pairs),
            "test": len(test_pairs),
            "eval": len(eval_pairs),
            "total": len(train_pairs) + len(test_pairs) + len(eval_pairs),
        }
    }

In [157]:
# Assuming you already have:
# index_paired = [(idx, damaged_path, undamaged_path), ...]
# and you want the default splits/paths.

train_test_pairs, eval_pairs = split_train_eval(
    indexed_pairs,
    train_eval_split=TRAIN_EVAL_SPLIT,
    seed=42
)

train_pairs, test_pairs = split_train_test(
    train_test_pairs,
    sub_train_test_split=SUB_TRAIN_TEST_SPLIT,
    seed=42
)

stats = move_split_files(
    train_pairs,
    test_pairs,
    eval_pairs,
    train_out=TRAIN_SET_PATH,
    test_out=TEST_SET_PATH,
    eval_out=EVAL_SET_PATH
)

print(stats)

{'train': 'splitsets/train_indexed_images', 'test': 'splitsets/test_indexed_images', 'eval': 'splitsets/eval_indexed_images', 'counts': {'train': 61, 'test': 26, 'eval': 21, 'total': 108}}


### Image tiling step

In [158]:
TILE_SIZE = 128 #tile size in pixels 

In [159]:
def tile_image_file_to(image_path, out_channel_dir, tile_size=TILE_SIZE):
    image_path = Path(image_path)
    out_channel_dir = Path(out_channel_dir)

    if not image_path.is_file():
        return

    idx = image_path.stem
    out_dir = out_channel_dir / idx
    out_dir.mkdir(parents=True, exist_ok=True)

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img = pad_to_multiple(img, tile_size=tile_size, fill=(255, 255, 255))

        w, h = img.size
        n_tiles_x = w // tile_size
        n_tiles_y = h // tile_size

        for y in range(n_tiles_y):
            for x in range(n_tiles_x):
                left = x * tile_size
                upper = y * tile_size
                tile = img.crop((left, upper, left + tile_size, upper + tile_size))
                tile.save(out_dir / f"{idx}_{x}_{y}.png", format="PNG")

In [160]:
def tile_splitsets_to(input_roots, output_roots, tile_size=TILE_SIZE):
    """
    Reads images from input_roots and writes tiled structure to output_roots,
    preserving the same overall structure:

      <output_root>/damaged/<idx>/<idx>_<x>_<y>.png
      <output_root>/undamaged/<idx>/<idx>_<x>_<y>.png

    input_roots can be a list like: [TRAIN_SET_PATH, TEST_SET_PATH, EVAL_SET_PATH]
    output_roots must match length 1:1 (same order).
    """
    if len(input_roots) != len(output_roots):
        raise ValueError("input_roots and output_roots must have the same length")

    for in_root, out_root in zip(map(Path, input_roots), map(Path, output_roots)):
        for channel in ("damaged", "undamaged"):
            in_channel_dir = in_root / channel
            out_channel_dir = Path(out_root) / channel
            out_channel_dir.mkdir(parents=True, exist_ok=True)

            if not in_channel_dir.exists():
                continue

            image_files = [p for p in in_channel_dir.iterdir() if p.is_file()]
            for img_path in image_files:
                tile_image_file_to(img_path, out_channel_dir, tile_size=tile_size)


In [161]:
TILED_TRAIN_SET_PATH = "tiledsets/train_indexed_images"
TILED_TEST_SET_PATH  = "tiledsets/test_indexed_images"
TILED_EVAL_SET_PATH  = "tiledsets/eval_indexed_images"

tile_splitsets_to(
    input_roots=[TRAIN_SET_PATH, TEST_SET_PATH, EVAL_SET_PATH],
    output_roots=[TILED_TRAIN_SET_PATH, TILED_TEST_SET_PATH, TILED_EVAL_SET_PATH],
    tile_size=TILE_SIZE
)